In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/csebuetnlp/normalizer"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-Uq", "bitsandbytes", "transformers", "peft", "pycocoevalcap"])

import json, os, random, re, unicodedata, shutil
import pandas as pd
from sklearn.model_selection import train_test_split
from normalizer import normalize

import torch
import torch.nn as nn
from collections import OrderedDict
from typing import Optional, Union, Tuple
from PIL import Image
from torch.utils.data import Dataset
from transformers import (
    Blip2Processor, Blip2PreTrainedModel, Blip2Config, AutoTokenizer, AutoModelForSeq2SeqLM, AutoConfig,
    TrainingArguments, Trainer, BitsAndBytesConfig
)
from transformers.models.blip_2.modeling_blip_2 import Blip2ForConditionalGenerationModelOutput
from torch.optim import AdamW
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings('ignore', message='.*unexpected keys.*')
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def bangla_postprocess(text):
    if not text or not isinstance(text, str): return text
    text = unicodedata.normalize('NFC', text)
    text = text.replace('\u200b', '').replace('\u200c', '').replace('\u200d', '').replace('\ufeff', '')
    text = re.sub(r'(?<![অ-হড়ঢ়য়ৎংঃঁ])[\u09BE-\u09CD\u09D7\u09BC]', '', text)
    text = normalize(text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    deduped = [words[0]] if words else []
    for w in words[1:]:
        if w != deduped[-1]: deduped.append(w)
    text = ' '.join(deduped)
    if text.endswith('.'): text = text[:-1] + '।'
    return text

# ── Config ──────────────────────────────────────────────────────────
OUT_DIR = "/kaggle/working/all_merged_splits"
os.makedirs(OUT_DIR, exist_ok=True)

class Config:
    # ── Dataset toggles & limits ────────────────────────────────────
    # Set to False to exclude the BanglaLekha dataset entirely
    use_banglalekha: bool = False

    # Maximum number of Flickr30k images to use.
    # Set to None to use all available images.
    flickr_limit: int = 15_000

    # ── Paths ────────────────────────────────────────────────────────
    images_bl = "/kaggle/input/datasets/hatecoder/bangla-lekha-image-captions/Bangla Lekha Image Captioning/images"
    images_fl = "/kaggle/input/datasets/adityajn105/flickr30k/Images/flickr30k_images"
    images_bn = "/kaggle/input/datasets/almominfaruk/bnaturebengali-image-captioning-dataset/Pictures"

    captions_bl = f"{OUT_DIR}/captions_banglalekha.txt"
    captions_fl = f"{OUT_DIR}/captions_flickr.txt"
    captions_bn = f"{OUT_DIR}/captions_bnature.txt"

    train_split = f"{OUT_DIR}/train.txt"
    val_split   = f"{OUT_DIR}/validation.txt"
    test_split  = f"{OUT_DIR}/test.txt"
    output_dir  = "/kaggle/working/blip2_bangla_all_merged_output"

    blip_checkpoint = "Salesforce/blip2-flan-t5-xl"
    bangla_model_id = "csebuetnlp/banglat5"

    max_len    = 128
    image_size = 224

    # Stage 1
    batch_size_s1            = 96
    gradient_accumulation_s1 = 1
    lr_stage1                = 2e-5
    epochs_stage1            = 5
    num_workers              = 16

    # Stage 2
    batch_size_s2            = 64
    gradient_accumulation_s2 = 2
    lr_qformer               = 1e-5
    lr_lora                  = 2e-5
    weight_decay             = 0.05
    epochs_stage2            = 30
    warmup_steps_s2          = 300

    lora_r       = 64
    lora_alpha   = 128
    lora_dropout = 0.05

    max_gen_length = 64
    num_beams      = 5
    length_penalty = 1.7
    device   = "cuda" if torch.cuda.is_available() else "cpu"
    use_8bit = True

config = Config()

# ── Data Prep ──────────────────────────────────────────
SEED = 42
def prepare_splits():
    print("Preparing splits...")

    # ── BanglaLekha (optional) ───────────────────────────────────────
    bl_train, bl_val, bl_data = [], [], []
    if config.use_banglalekha:
        SRC_JSON_BL = (
            "/kaggle/input/datasets/hatecoder/bangla-lekha-image-captions"
            "/Bangla Lekha Image Captioning/captions.json"
        )
        N_TRAIN_BL = 7323
        with open(SRC_JSON_BL, 'r', encoding='utf-8') as f:
            bl_data = json.load(f)
        random.seed(SEED); random.shuffle(bl_data)
        bl_train, bl_val = bl_data[:N_TRAIN_BL], bl_data[N_TRAIN_BL:]
        print(f"  BanglaLekha  → train={len(bl_train)}, val={len(bl_val)}")
    else:
        print("  BanglaLekha  → SKIPPED (use_banglalekha=False)")

    # ── Flickr30k (with optional image limit) ────────────────────────
    CSV_FL = (
        "/kaggle/input/datasets/nayeemuzzaman90/banglaview-a-bangla-image-captioning-dataset"
        "/banglaview_dataset.csv"
    )
    fl_df = pd.read_csv(CSV_FL)
    fl_df.columns = fl_df.columns.str.strip().str.lower()
    flickr_dict = {}
    for _, row in fl_df.iterrows():
        fname = str(row['caption_id']).split('#')[0].strip()
        if not fname.endswith('.jpg'): fname += '.jpg'
        flickr_dict.setdefault(fname, []).append(str(row['bengali_caption']).strip())

    fl_all = sorted(flickr_dict.keys())

    # Apply image limit if set
    if config.flickr_limit is not None:
        random.seed(SEED)
        fl_all = random.sample(fl_all, min(config.flickr_limit, len(fl_all)))
        fl_all = sorted(fl_all)       
        print(f"  Flickr30k    → using {len(fl_all)} / {len(flickr_dict)} images "
              f"(flickr_limit={config.flickr_limit})")
    else:
        print(f"  Flickr30k    → using all {len(fl_all)} images")

    fl_train_list, fl_temp = train_test_split(fl_all, test_size=0.2, random_state=SEED)
    fl_val_list, fl_test_list = train_test_split(fl_temp, test_size=0.5, random_state=SEED)

    # ── Bnature ──────────────────────────────────────────────────────
    BN_CAPTIONS = (
        "/kaggle/input/datasets/almominfaruk/bnaturebengali-image-captioning-dataset"
        "/caption/caption.txt"
    )
    BN_TRAIN = (
        "/kaggle/input/datasets/almominfaruk/bnaturebengali-image-captioning-dataset"
        "/caption/train.txt"
    )
    BN_VAL = (
        "/kaggle/input/datasets/almominfaruk/bnaturebengali-image-captioning-dataset"
        "/caption/validation.txt"
    )
    BN_TEST = (
        "/kaggle/input/datasets/almominfaruk/bnaturebengali-image-captioning-dataset"
        "/caption/test.txt"
    )
    def load_txt(p):
        with open(p, 'r', encoding='utf-8') as f:
            return [l.strip() for l in f if l.strip()]

    bn_train_list = load_txt(BN_TRAIN)
    bn_val_list   = load_txt(BN_VAL)
    bn_test_list  = load_txt(BN_TEST)
    print(f"  Bnature      → train={len(bn_train_list)}, val={len(bn_val_list)}, test={len(bn_test_list)}")

    # ── Write caption files ──────────────────────────────────────────
    if config.use_banglalekha:
        with open(config.captions_bl, 'w', encoding='utf-8') as f:
            for entry in bl_data:
                for cap in entry['caption']:
                    f.write(f"{entry['filename']}\t{cap}\n")

    with open(config.captions_fl, 'w', encoding='utf-8') as f:
        for fname in fl_all:                       # only the (possibly limited) subset
            for cap in flickr_dict[fname]:
                f.write(f"{fname}\t{cap}\n")

    shutil.copy(BN_CAPTIONS, config.captions_bn)

    # ── Write split files ────────────────────────────────────────────
    with open(config.train_split, 'w', encoding='utf-8') as f:
        if config.use_banglalekha:
            for entry in bl_train: f.write("BL:" + entry['filename'] + '\n')
        for fname in fl_train_list: f.write("FL:" + fname + '\n')
        for fname in bn_train_list: f.write("BN:" + fname + '\n')

    with open(config.val_split, 'w', encoding='utf-8') as f:
        if config.use_banglalekha:
            for entry in bl_val: f.write("BL:" + entry['filename'] + '\n')
        for fname in fl_val_list:  f.write("FL:" + fname + '\n')
        for fname in bn_val_list:  f.write("BN:" + fname + '\n')

    with open(config.test_split, 'w', encoding='utf-8') as f:
        if config.use_banglalekha:
            for entry in bl_val: f.write("BL:" + entry['filename'] + '\n')
        for fname in fl_test_list: f.write("FL:" + fname + '\n')
        for fname in bn_test_list: f.write("BN:" + fname + '\n')

    print(f"Splits saved to {OUT_DIR}")

# ── DataLoader ──────────────────────────────────────────
def load_merged_caption_maps():
    bl_captions, fl_captions, bn_captions = {}, {}, {}

    # BanglaLekha — only load if the file was written
    if config.use_banglalekha and os.path.exists(config.captions_bl):
        with open(config.captions_bl, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t', 1)
                if len(parts) == 2:
                    fname = parts[0].replace(' ', '')
                    if not fname.endswith('.png'): fname += '.png'
                    bl_captions.setdefault(fname, []).append(normalize(parts[1]) or "")

    with open(config.captions_fl, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t', 1)
            if len(parts) == 2:
                fname = parts[0].replace(' ', '')
                if not fname.endswith('.jpg'): fname += '.jpg'
                fl_captions.setdefault(fname, []).append(normalize(parts[1]) or "")

    with open(config.captions_bn, 'r', encoding='utf-8') as f:
        for line in f:
            parts = (line.strip().split('\t', 1)
                     if '\t' in line.strip()
                     else line.strip().split(None, 1))
            if len(parts) == 2:
                fname = parts[0].replace(' ', '')
                if not fname.endswith('.jpg'): fname += '.jpg'
                bn_captions.setdefault(fname, []).append(normalize(parts[1]) or "")

    return bl_captions, fl_captions, bn_captions

def process_merged_splits(split_file, bl_captions, fl_captions, bn_captions, for_eval=False):
    with open(split_file, 'r', encoding='utf-8') as f:
        lines = [l.strip() for l in f if l.strip()]
    matched = []
    for line in lines:
        src, fname = line[:2], line[3:].replace(' ', '')
        if src == "BL":
            # Skip silently if BanglaLekha is disabled
            if not config.use_banglalekha:
                continue
            if not fname.endswith('.png'): fname += '.png'
            if fname in bl_captions:
                if for_eval: matched.append(("BL", fname, bl_captions[fname]))
                else:
                    for cp in bl_captions[fname]: matched.append(("BL", fname, cp))
        elif src == "FL":
            if not fname.endswith('.jpg'): fname += '.jpg'
            if fname in fl_captions:
                if for_eval: matched.append(("FL", fname, fl_captions[fname]))
                else:
                    for cp in fl_captions[fname]: matched.append(("FL", fname, cp))
        elif src == "BN":
            if not fname.endswith('.jpg'): fname += '.jpg'
            if fname in bn_captions:
                if for_eval: matched.append(("BN", fname, bn_captions[fname]))
                else:
                    for cp in bn_captions[fname]: matched.append(("BN", fname, cp))
    return matched

class BanglaMergedDataset(Dataset):
    def __init__(self, data_list, processor, tokenizer, is_eval=False):
        self.data = data_list
        self.processor = processor
        self.tokenizer = tokenizer
        self.is_eval = is_eval
        self.templates = ["বাংলায় ক্যাপশন:", "ছবির বর্ণনা:", "বর্ণনা:", "চিত্রে দেখা যাচ্ছে যে: "]

    def __len__(self): return len(self.data)

    def _get_path(self, source, filename):
        if source == "BL": return os.path.join(config.images_bl, filename)
        elif source == "FL": return os.path.join(config.images_fl, filename)
        elif source == "BN": return os.path.join(config.images_bn, filename)
        return ""

    def __getitem__(self, idx):
        if self.is_eval: source, filename, captions = self.data[idx]
        else: source, filename, caption = self.data[idx]

        img_path = self._get_path(source, filename)
        try: image = Image.open(img_path).convert('RGB')
        except: image = Image.new('RGB', (config.image_size, config.image_size))
        pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze(0)

        instruction = "বাংলায় ক্যাপশন:" if self.is_eval else random.choice(self.templates)
        text_enc = self.tokenizer(instruction, padding="max_length", max_length=config.max_len,
                                  truncation=True, return_tensors="pt")
        target = captions[0] if self.is_eval else caption
        tgt_enc = self.tokenizer(target, padding="max_length", max_length=config.max_len,
                                 truncation=True, return_tensors="pt")

        out = {
            'pixel_values': pixel_values,
            'input_ids': text_enc['input_ids'].squeeze(0),
            'attention_mask': text_enc['attention_mask'].squeeze(0),
            'labels': tgt_enc['input_ids'].squeeze(0)
        }
        if self.is_eval:
            out['reference_captions'] = captions
            out['filename'] = f"{source}_{filename}"
        return out

class MultimodalCollator:
    def __init__(self, tokenizer): self.tokenizer = tokenizer
    def __call__(self, batch):
        pixel_values   = torch.stack([item['pixel_values']   for item in batch])
        input_ids      = torch.stack([item['input_ids']      for item in batch])
        attention_mask = torch.stack([item['attention_mask'] for item in batch])
        labels         = torch.stack([item['labels']         for item in batch])
        labels[labels == self.tokenizer.pad_token_id] = -100
        batch_out = {
            'pixel_values': pixel_values, 'input_ids': input_ids,
            'attention_mask': attention_mask, 'labels': labels
        }
        if 'reference_captions' in batch[0]:
            batch_out['reference_captions'] = [item['reference_captions'] for item in batch]
        if 'filename' in batch[0]:
            batch_out['filename'] = [item['filename'] for item in batch]
        return batch_out

# ── Model ──────────────────────────────────────────
class BanglaBLIP(Blip2PreTrainedModel):
    def __init__(self, blip_pretrained="Salesforce/blip2-flan-t5-xl",
                 bangla_lm_pretrained="csebuetnlp/banglat5", load_8bit=True,
                 freeze_vit=True, freeze_qformer=True, freeze_lm=True,
                 freeze_projection=False, use_lora=False):
        from transformers import Blip2ForConditionalGeneration
        blip2_model = Blip2ForConditionalGeneration.from_pretrained(blip_pretrained, torch_dtype=torch.float16)

        cfg = blip2_model.config
        bangla_config = AutoConfig.from_pretrained(bangla_lm_pretrained)
        cfg.text_config = bangla_config
        super().__init__(cfg)

        self.vision_model  = blip2_model.vision_model
        self.qformer       = blip2_model.qformer
        self.query_tokens  = blip2_model.query_tokens

        self.language_projection = nn.Sequential(
            nn.Linear(self.qformer.config.hidden_size, bangla_config.d_model),
            nn.GELU(),
            nn.Linear(bangla_config.d_model, bangla_config.d_model))
        for layer in self.language_projection:
            if isinstance(layer, nn.Linear):
                nn.init.normal_(layer.weight, std=0.02)
                nn.init.zeros_(layer.bias)

        self.language_projection_ln = nn.LayerNorm(bangla_config.d_model)
        self.language_projection_ln.requires_grad_(True)

        del blip2_model.language_model; del blip2_model
        torch.cuda.empty_cache()

        rank = int(os.environ.get("LOCAL_RANK", 0))
        self.llm_cast_dtype = torch.bfloat16

        if load_8bit:
            bnb_config = BitsAndBytesConfig(load_in_8bit=True)
            self.language_model = AutoModelForSeq2SeqLM.from_pretrained(
                bangla_lm_pretrained, dtype="auto",
                quantization_config=bnb_config, device_map={"": rank})
        else:
            self.language_model = AutoModelForSeq2SeqLM.from_pretrained(
                bangla_lm_pretrained, dtype="auto", device_map={"": rank})

        if freeze_vit:
            for param in self.vision_model.parameters(): param.requires_grad = False
        if freeze_qformer:
            self.query_tokens.requires_grad = False
            for param in self.qformer.parameters(): param.requires_grad = False
        if freeze_lm:
            for param in self.language_model.parameters(): param.requires_grad = False
        if freeze_projection:
            for param in self.language_projection.parameters(): param.requires_grad = False

        if use_lora:
            self.language_model = prepare_model_for_kbit_training(
                self.language_model, use_gradient_checkpointing=True)
            lora_cfg = LoraConfig(
                r=config.lora_r, lora_alpha=config.lora_alpha,
                target_modules=["q", "k", "v", "o", "wi_0", "wi_1", "wo"],
                lora_dropout=config.lora_dropout, bias="none", task_type="SEQ_2_SEQ_LM")
            self.language_model = get_peft_model(self.language_model, lora_cfg)

        self._use_lora = bool(use_lora)

    def state_dict(self, *args, destination=None, prefix='', keep_vars=False):
        if destination is None:
            destination = OrderedDict(); destination._metadata = OrderedDict()
        local_metadata = dict(version=self._version)
        if hasattr(destination, "_metadata"): destination._metadata[prefix[:-1]] = local_metadata
        self._save_to_state_dict(destination, prefix, keep_vars)
        for name, module in self._modules.items():
            if module is None or name == "vision_model": continue
            if name == "language_model":
                if self._use_lora:
                    lora_sd = module.state_dict(destination=None, prefix='', keep_vars=keep_vars)
                    for k, v in lora_sd.items():
                        if "lora_" in k: destination[prefix + name + "." + k] = v
                continue
            module.state_dict(destination=destination, prefix=prefix + name + ".", keep_vars=keep_vars)
        for hook in self._state_dict_hooks.values():
            hook_result = hook(self, destination, prefix, local_metadata)
            if hook_result is not None: destination = hook_result
        return destination

    def get_input_embeddings(self): return self.language_model.get_input_embeddings()
    def set_input_embeddings(self, value): self.language_model.set_input_embeddings(value)

    def forward(self, pixel_values, input_ids, attention_mask=None, decoder_input_ids=None,
                decoder_attention_mask=None, output_attentions=None, output_hidden_states=None,
                labels=None, return_dict=None, **kwargs):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        num_images = 1
        if len(pixel_values.shape) == 5:
            num_images = pixel_values.shape[1]
            pixel_values = pixel_values.view(
                pixel_values.shape[0] * pixel_values.shape[1], *pixel_values.shape[2:])
        vision_outputs = self.vision_model(
            pixel_values=pixel_values, output_attentions=output_attentions,
            output_hidden_states=output_hidden_states, return_dict=return_dict)
        image_embeds = vision_outputs[0]
        image_attention_mask = torch.ones(
            image_embeds.size()[:-1], dtype=torch.long, device=image_embeds.device)
        query_tokens = self.query_tokens.expand(image_embeds.shape[0], -1, -1)
        query_outputs = self.qformer(
            query_embeds=query_tokens, encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_attention_mask, output_attentions=output_attentions,
            output_hidden_states=output_hidden_states, return_dict=return_dict)
        query_output = query_outputs[0]
        if num_images > 1:
            query_output = query_output.view(input_ids.shape[0], -1, query_output.shape[2])
        language_model_inputs = self.language_projection(query_output)
        language_model_inputs = self.language_projection_ln(language_model_inputs)
        language_model_attention_mask = torch.ones(
            language_model_inputs.size()[:-1], dtype=torch.long,
            device=language_model_inputs.device)
        with torch.amp.autocast("cuda", dtype=self.llm_cast_dtype):
            lm_embedding  = self.language_model.get_input_embeddings()
            inputs_embeds = lm_embedding(input_ids)
            inputs_embeds = torch.cat(
                [language_model_inputs, inputs_embeds.to(language_model_inputs.device)], dim=1)
            if attention_mask is None: attention_mask = torch.ones_like(input_ids)
            attention_mask = torch.cat(
                [language_model_attention_mask,
                 attention_mask.to(language_model_attention_mask.device)], dim=1)
            outputs = self.language_model(
                inputs_embeds=inputs_embeds, attention_mask=attention_mask,
                decoder_input_ids=decoder_input_ids,
                decoder_attention_mask=decoder_attention_mask,
                output_attentions=output_attentions, output_hidden_states=output_hidden_states,
                return_dict=return_dict, labels=labels)
            loss   = outputs.loss   if return_dict else outputs[0]
            logits = outputs.logits if return_dict else outputs[1]
        if not return_dict:
            output = (logits, vision_outputs, query_outputs, outputs)
            return ((loss,) + output) if loss is not None else output
        return Blip2ForConditionalGenerationModelOutput(
            loss=loss, logits=logits, vision_outputs=vision_outputs,
            qformer_outputs=query_outputs, language_model_outputs=outputs)

    @torch.no_grad()
    def generate(self, pixel_values, input_ids=None, attention_mask=None, **generate_kwargs):
        num_images = 1
        orig_batch_size = pixel_values.shape[0]
        if len(pixel_values.shape) == 5:
            orig_batch_size = pixel_values.shape[0]
            num_images = pixel_values.shape[1]
            pixel_values = pixel_values.view(
                pixel_values.shape[0] * pixel_values.shape[1], *pixel_values.shape[2:])
        image_embeds = self.vision_model(pixel_values, return_dict=True).last_hidden_state
        image_attention_mask = torch.ones(
            image_embeds.size()[:-1], dtype=torch.long, device=image_embeds.device)
        query_tokens = self.query_tokens.expand(image_embeds.shape[0], -1, -1)
        query_outputs = self.qformer(
            query_embeds=query_tokens, encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_attention_mask, return_dict=True)
        query_output = query_outputs.last_hidden_state
        if num_images > 1:
            query_output = query_output.view(orig_batch_size, -1, query_output.shape[2])
        language_model_inputs = self.language_projection(query_output)
        language_model_inputs = self.language_projection_ln(language_model_inputs)
        language_attention_mask = torch.ones(
            language_model_inputs.size()[:-1], dtype=torch.long,
            device=language_model_inputs.device)
        if input_ids is None:
            input_ids = (torch.LongTensor([[self.config.text_config.bos_token_id]])
                         .repeat(orig_batch_size, 1).to(image_embeds.device))
        if attention_mask is None: attention_mask = torch.ones_like(input_ids)
        attention_mask = torch.cat([language_attention_mask, attention_mask], dim=1)
        with torch.amp.autocast("cuda", dtype=self.llm_cast_dtype):
            lm_embedding  = self.language_model.get_input_embeddings()
            inputs_embeds = lm_embedding(input_ids)
            inputs_embeds = torch.cat(
                [language_model_inputs, inputs_embeds.to(language_model_inputs.device)], dim=1)
            outputs = self.language_model.generate(
                inputs_embeds=inputs_embeds, attention_mask=attention_mask, **generate_kwargs)
        return outputs

# ── Evaluate ──────────────────────────────────────────
import csv
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction, sentence_bleu

def evaluate_with_bleu(model, eval_dataset, tokenizer, num_samples=None, csv_path="bleu_results.csv"):
    model.eval()
    device = config.device
    predictions, references, rows = [], [], []
    indices = (range(len(eval_dataset))
               if num_samples is None
               else range(min(num_samples, len(eval_dataset))))
    smoothie = SmoothingFunction().method1

    for idx in indices:
        sample = eval_dataset[idx]
        pv = sample['pixel_values'].unsqueeze(0).to(device)
        ii = sample['input_ids'].unsqueeze(0).to(device)
        am = sample['attention_mask'].unsqueeze(0).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                pixel_values=pv, input_ids=ii, attention_mask=am,
                max_length=config.max_gen_length, num_beams=config.num_beams,
                length_penalty=config.length_penalty, no_repeat_ngram_size=3,
                repetition_penalty=1.5)

        generated = bangla_postprocess(
            tokenizer.decode(output_ids[0], skip_special_tokens=True))
        gen_tok  = generated.split()
        ref_caps = [bangla_postprocess(r) for r in sample['reference_captions']]
        ref_tok  = [r.split() for r in ref_caps]

        predictions.append(gen_tok)
        references.append(ref_tok)
        rows.append({
            "filename": sample.get("filename", f"image_{idx}"),
            "generated_caption": generated,
            "reference_captions": " ||| ".join(ref_caps),
            "sentence_bleu1": sentence_bleu(ref_tok, gen_tok, weights=(1,0,0,0), smoothing_function=smoothie),
            "sentence_bleu2": sentence_bleu(ref_tok, gen_tok, weights=(0.5,0.5,0,0), smoothing_function=smoothie),
            "sentence_bleu3": sentence_bleu(ref_tok, gen_tok, weights=(0.33,0.33,0.33,0), smoothing_function=smoothie),
            "sentence_bleu4": sentence_bleu(ref_tok, gen_tok, weights=(0.25,0.25,0.25,0.25), smoothing_function=smoothie)
        })

    bleu1 = corpus_bleu(references, predictions, weights=(1,0,0,0), smoothing_function=smoothie)
    bleu2 = corpus_bleu(references, predictions, weights=(0.5,0.5,0,0), smoothing_function=smoothie)
    bleu3 = corpus_bleu(references, predictions, weights=(0.33,0.33,0.33,0), smoothing_function=smoothie)
    bleu4 = corpus_bleu(references, predictions, weights=(0.25,0.25,0.25,0.25), smoothing_function=smoothie)

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader(); writer.writerows(rows)
    print(f"\nBLEU-4: {bleu4:.4f} (see {csv_path})")
    model.train()

# ── Training Flow ──────────────────────────────────────────
def get_model_and_processor():
    processor = Blip2Processor.from_pretrained(config.blip_checkpoint)
    tokenizer = AutoTokenizer.from_pretrained(config.bangla_model_id, use_fast=False)
    model = BanglaBLIP(
        blip_pretrained=config.blip_checkpoint,
        bangla_lm_pretrained=config.bangla_model_id,
        load_8bit=config.use_8bit,
        freeze_vit=True, freeze_qformer=True, freeze_lm=True,
        freeze_projection=False, use_lora=False)
    return model, processor, tokenizer

def run_stage1_warmup(model, dataset, val_dataset, collator):
    for param in model.parameters(): param.requires_grad = False
    for param in model.language_projection.parameters(): param.requires_grad = True

    args = TrainingArguments(
        output_dir=f"{config.output_dir}/stage1",
        num_train_epochs=config.epochs_stage1,
        per_device_train_batch_size=config.batch_size_s1,
        gradient_accumulation_steps=config.gradient_accumulation_s1,
        learning_rate=config.lr_stage1, warmup_steps=10,
        logging_strategy="steps", logging_steps=300,
        save_steps=3000, save_total_limit=1, fp16=False,
        remove_unused_columns=False, dataloader_num_workers=config.num_workers,
        eval_strategy="steps" if val_dataset else "no",
        eval_steps=1500 if val_dataset else None,
        report_to="none", max_grad_norm=1.0)
    trainer = Trainer(model=model, args=args, train_dataset=dataset,
                      eval_dataset=val_dataset, data_collator=collator)
    trainer.train()
    model.save_pretrained(f"{config.output_dir}/stage1_checkpoint")
    return model

def run_stage2_realignment(model, dataset, val_dataset, collator, tokenizer):
    for param in model.qformer.parameters(): param.requires_grad = True
    model.query_tokens.requires_grad = True
    model.language_model = prepare_model_for_kbit_training(model.language_model)
    lora_cfg = LoraConfig(
        r=config.lora_r, lora_alpha=config.lora_alpha,
        target_modules=["q", "k", "v", "o", "wi_0", "wi_1", "wo"],
        lora_dropout=config.lora_dropout, bias="none", task_type="SEQ_2_SEQ_LM")
    model.language_model = get_peft_model(model.language_model, lora_cfg)

    lora_plus_ratio = 16.0
    optimizer_grouped_parameters = [
        {"params": [p for n, p in model.named_parameters() if "lora_A" in n], "lr": config.lr_lora},
        {"params": [p for n, p in model.named_parameters() if "lora_B" in n], "lr": config.lr_lora * lora_plus_ratio},
        {"params": [p for n, p in model.named_parameters() if "language_projection" in n], "lr": config.lr_qformer},
        {"params": list(model.qformer.parameters()) + [model.query_tokens], "lr": config.lr_qformer},
    ]

    args = TrainingArguments(
        output_dir=f"{config.output_dir}/stage2",
        num_train_epochs=config.epochs_stage2,
        per_device_train_batch_size=config.batch_size_s2,
        gradient_accumulation_steps=config.gradient_accumulation_s2,
        learning_rate=config.lr_lora, weight_decay=config.weight_decay,
        warmup_steps=config.warmup_steps_s2,
        logging_strategy="steps", logging_steps=300,
        save_steps=6000, save_total_limit=1, fp16=True,
        remove_unused_columns=False, dataloader_num_workers=config.num_workers,
        eval_strategy="steps" if val_dataset else "no",
        eval_steps=2000 if val_dataset else None,
        gradient_checkpointing=True, report_to="none", max_grad_norm=1.0)
    trainer = Trainer(
        model=model, args=args, train_dataset=dataset,
        eval_dataset=val_dataset, data_collator=collator,
        optimizers=(AdamW(optimizer_grouped_parameters), None))
    trainer.train()
    model.save_pretrained(f"{config.output_dir}/final_model")
    model.language_model.save_pretrained(f"{config.output_dir}/final_model/lora_adapter")
    tokenizer.save_pretrained(f"{config.output_dir}/final_model")
    return model

def run_training():
    prepare_splits()
    bl_caps, fl_caps, bn_caps = load_merged_caption_maps()
    train_data = process_merged_splits(config.train_split, bl_caps, fl_caps, bn_caps, for_eval=False)
    val_data   = process_merged_splits(config.val_split,   bl_caps, fl_caps, bn_caps, for_eval=True)
    test_data  = process_merged_splits(config.test_split,  bl_caps, fl_caps, bn_caps, for_eval=True)
    print(f"train={len(train_data)}, val={len(val_data)}, test={len(test_data)}")
    test_data = test_data[:100]

    model, processor, tokenizer = get_model_and_processor()
    train_dataset = BanglaMergedDataset(train_data, processor, tokenizer, is_eval=False)
    val_dataset   = BanglaMergedDataset(val_data,   processor, tokenizer, is_eval=True) if val_data   else None
    test_dataset  = BanglaMergedDataset(test_data,  processor, tokenizer, is_eval=True) if test_data  else None
    collator = MultimodalCollator(tokenizer)

    model = run_stage1_warmup(model, train_dataset, val_dataset, collator)
    model = run_stage2_realignment(model, train_dataset, val_dataset, collator, tokenizer)
    if test_dataset: evaluate_with_bleu(model, test_dataset, tokenizer, num_samples=10)

if __name__ == "__main__":
    run_training()

import matplotlib.pyplot as plt, numpy as np
from IPython.display import display
import nltk
csv_path = "bleu_results.csv"
nltk.download('wordnet')
nltk.download('omw-1.4')  # extended WordNet, recommended alongside wordnet

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df["generated_caption"] = df["generated_caption"].fillna("").astype(str)
    df["reference_captions"] = df["reference_captions"].fillna("").astype(str)

    from nltk.translate.bleu_score import corpus_bleu
    from nltk.translate.meteor_score import meteor_score as nltk_meteor
    from pycocoevalcap.cider.cider import Cider as CiderScorer

    refs_nested, hyps = [], []
    for _, row in df.iterrows():
        refs = row["reference_captions"].split(" ||| ")
        refs_nested.append([r.split() for r in refs])
        hyps.append(row["generated_caption"].split())

    corpus_scores = {
        "BLEU-1": corpus_bleu(refs_nested, hyps, weights=(1, 0, 0, 0)),
        "BLEU-2": corpus_bleu(refs_nested, hyps, weights=(0.5, 0.5, 0, 0)),
        "BLEU-3": corpus_bleu(refs_nested, hyps, weights=(1/3, 1/3, 1/3, 0)),
        "BLEU-4": corpus_bleu(refs_nested, hyps, weights=(0.25, 0.25, 0.25, 0.25)),
    }

    # METEOR — averaged over all samples
    meteor_scores = [
        nltk_meteor([r.split() for r in row["reference_captions"].split(" ||| ")],
                    row["generated_caption"].split())
        for _, row in df.iterrows()
    ]
    corpus_scores["METEOR"] = sum(meteor_scores) / len(meteor_scores)

    # CIDEr
    gts = {i: row["reference_captions"].split(" ||| ") for i, row in df.iterrows()}
    res = {i: [row["generated_caption"]] for i, row in df.iterrows()}
    cider_score, _ = CiderScorer().compute_score(gts, res)
    corpus_scores["CIDEr"] = cider_score

    summary_df = pd.DataFrame({
        "Metric": list(corpus_scores.keys()),
        "Score": [f"{v:.4f}" for v in corpus_scores.values()]
    })
    display(
        summary_df.style
        .set_caption("BanglaBLIP — All 3 DB Merged")
        .set_properties(**{"text-align": "center"})
        .hide(axis="index")
    )

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 194.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 151.5 MB/s eta 0:00:00
Preparing splits...
  BanglaLekha  → SKIPPED (use_banglalekha=False)
  Flickr30k    → using 15000 / 31783 images (flickr_limit=15000)
  Bnature      → train=6000, val=1000, test=1000
Splits saved to /kaggle/working/all_merged_splits
train=89308, val=2500, test=2500


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


BLEU-4: 0.0973 (see bleu_results.csv)


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


Metric,Score
BLEU-1,0.4538
BLEU-2,0.2812
BLEU-3,0.1587
BLEU-4,0.0973
METEOR,0.2738
CIDEr,0.3621
